# islide — M1 interactive demo

The canvas-based viewer. `islide.SlideViewer` is a custom Jupyter widget:
Python owns the slide (open, level selection, one-read-per-viewport, tile
cache, JPEG encode); the JS `SlideView` owns the canvas, the pan/zoom
transform, and the mouse. The minimap, zoom/fit/1:1 toolbar, and the
`status` line are part of the view.

> **Note:** the interactive canvas requires the JupyterLab/notebook
> extension (`frontend/`) to be built and installed. In environments
> without it the cells below still run (open + render happen in the
> kernel) but draw no canvas.

In [ ]:
from islide import SlideViewer

# Test slide (data/ is gitignored): download data/CMU-1.tiff from
# https://openslide.cs.cmu.edu/download/openslide-testdata/Generic-TIFF/CMU-1.tiff
v = SlideViewer("data/CMU-1.tiff")
v

The slide opens on a **background thread** (the kernel stays responsive).
Block until it's ready, then inspect the synced metadata:

In [ ]:
v.wait()
v.meta

Navigate with the mouse: **scroll** to zoom at the cursor, **drag** to
pan, or use the **minimap** and the `fit / 1:1` toolbar. Pan/zoom is
applied instantly in the view
(local transform) and the new viewport is synced back to Python (debounced),
which then fetches any missing tiles.

## Programmatic control

The same API drives the view from Python. These update the `viewport`
trait; the view re-centers on the next sync.

In [ ]:
v.center_on(23000, 16457)   # middle of the slide
v.set_zoom(2.0)

In [ ]:
# What's on screen right now (level-0 px)?
bbox = v.viewport_bbox()
bbox

In [ ]:
# Programmatic reading of the current view (feeds M2 rubber-band / last_region):
crop = v.read_crop(bbox)
crop

## Tiles & geometry

`v.tiles` maps `"level:tx:ty"` → JPEG data URL; `v.tile_geo` maps the same
key → `[level, ox, oy, cw, ch]` (absolute level-pixel crop origin + size).
The view reprojects this geometry under its local transform, so panning
stays smooth between Python round-trips.

In [ ]:
import json
keys = list(v.tiles)
print(f"{len(keys)} tiles for the current viewport")
for k in keys[:4]:
    print(f"  {k}: geo={v.tile_geo[k]}  url_len={len(v.tiles[k])}")

## Viewport height

`canvas_h` is a synced trait: the on-screen height of the canvas
viewport (CSS px). Set it at construction (`SlideViewer(path, canvas_h=900)`)
or at runtime — the view applies the new height, its ResizeObserver
syncs the resized viewport back to Python, and Python re-plans the
tiles at the new size.

In [ ]:
v.canvas_h = 800
v.canvas_h

Close (joins the open thread, releases the OpenSlide handle).

In [ ]:
v.close()